# NLP Robot Command Parser

## 0. Setup
Prepares the environment. The repository is cloned from GitHub, and the SCAN dataset is downloaded. Required dependencies are installed from requirements.txt. The configuration file (config.json) is loaded to set model and training parameters.

In [ ]:
# Clone repo and move into it 
!git clone https://github.com/PetraMicanovic/nlp-robot-command-parser.git
%cd nlp-robot-command-parser

In [ ]:
!git clone https://github.com/brendenlake/SCAN.git data/scan

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import json, sys, torch, random, numpy as np
sys.path.insert(0, '.')   # makes src/ importable

with open('config.json') as f:
    cfg = json.load(f)

SEED = cfg['training']['seed']
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)       
torch.backends.cudnn.deterministic = True  
torch.backends.cudnn.benchmark = False 

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {cfg["model"]["name"]}')

## 1. Load data
Loads the SCAN dataset using the load_scan function. The data is split into training and test sets based on the configuration. 

It can be loaded in either English or Serbian depending on the selected language (`cfg['data']['lang']`). When Serbian is selected, commands and actions are automatically translated.

Basic dataset informations are displayed.

In [ ]:
from src.data.load_data import load_scan

language = cfg['data']['lang'] # 'sr' or 'en'
split = cfg['data']['scan_split'] 

train_data, test_data = load_scan(
    split     = split,
    base_path = cfg['data']['scan_base_path'],
    lang = language,
)

print(f'Language : {language}')
print(f'Train examples : {len(train_data)}')
print(f'Test  examples : {len(test_data)}')
print(f'First example  : {train_data[0]}')

### 1.1. Dataset statistics

In [ ]:
from src.data.translate_scan import print_stats

print_stats(train_data, test_data)

## 2. Preprocessing (tokenization)

This step prepares the dataset for sequence-to-sequence training with T5. A tokenizer is loaded based on the selected model, and the raw data is converted into Hugging Face `Dataset` format.

The dataset is then tokenized by adding a task-specific prefix, encoding commands and actions, and preparing labels for training. Tokenization is applied separately to the training and test splits using a `DatasetDict`.

In [ ]:
from src.data.preprocess import get_tokenizer, to_hf_dataset, tokenize_dataset
from datasets import DatasetDict

tokenizer = get_tokenizer(cfg['model']['name'])

raw_dataset = DatasetDict({
    'train': to_hf_dataset(train_data),
    'test' : to_hf_dataset(test_data),
})

tokenized_dataset = DatasetDict({
    split: tokenize_dataset(
        raw_dataset[split], tokenizer,
        prefix         = cfg['model']['prefix'],
        max_input_len  = cfg['model']['max_input_len'],
        max_target_len = cfg['model']['max_target_len'],
    )
    for split in ('train', 'test')
})

print('Tokenization complete.')
print(tokenized_dataset)

## 3. Load pretrained T5 model

The pretrained T5 model is loaded using the configuration parameters and moved to the target device (CPU/GPU).

In [ ]:
from src.models.t5_model import load_model

model = load_model(cfg['model']['name'], DEVICE)

## 4. Training

The Seq2SeqTrainer is initialized using the model, tokenizer, tokenized dataset and training configuration. Training is then started with `trainer.train()`, which handles the full training loop, including evaluation and checkpoint saving. 

After training, the final model and tokenizer are saved locally to the `checkpoints/final` directory using `save_pretrained()`.

If training is performed in Google Colab, the saved model can be copied to Google Drive for persistence. This step is optional and environment-dependent, as local training setups do not require external storage.

In [ ]:
from src.training.trainer import build_trainer

trainer = build_trainer(
    model             = model,
    tokenizer         = tokenizer,
    tokenized_dataset = tokenized_dataset,
    training_cfg      = cfg['training'],
    device_fp16       = (DEVICE == 'cuda'),
)

trainer.train()

In [ ]:
# Save final model at local checkpoints/ directory 
SAVE_PATH = 'checkpoints/final'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model saved to: {SAVE_PATH}')

In [ ]:
# Copy on Google Drive 
from google.colab import drive
drive.mount('/content/drive')
!cp -r {SAVE_PATH} /content/drive/MyDrive/nlp-robot-command-parser/

## 5. Load trained model 

The trained model and tokenizer are loaded from `checkpoints/final` using `from_pretrained()`. 

If running in Google Colab, Google Drive is mounted and the path is automatically redirected to the Drive location for persistent storage.

The code checks whether the model path exists and raises an error if not. The model is then moved to the selected device (`CPU` or `GPU`).

This step allows directly using a previously trained model, skipping steps 3 and 4 (model initialization and training).

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import os

LOAD_PATH = 'checkpoints/final'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    LOAD_PATH = '/content/drive/MyDrive/nlp-robot-command-parser/final'
except ImportError:
    pass  

if not os.path.exists(LOAD_PATH):
    raise FileNotFoundError(
        f'Model not found on path: {LOAD_PATH}\n'
    )

model     = T5ForConditionalGeneration.from_pretrained(LOAD_PATH)
tokenizer = T5Tokenizer.from_pretrained(LOAD_PATH)
model     = model.to(DEVICE)

print(f' Model loaded from: {LOAD_PATH}')
print(f' Device: {DEVICE}')

## 6. Decoding examples

This section demonstrates model predictions on example inputs using constrained decoding. 
The generation process is restricted to a predefined set of valid actions by masking invalid tokens.

In [ ]:
from src.models.t5_model import predict
from src.data.translate_scan import build_bad_word_ids, get_valid_actions

valid_actions = get_valid_actions(lang=language)
bad_word_ids  = build_bad_word_ids(tokenizer, valid_actions)

# Examples in Serbian (if language='sr') or English (if language='en')
examples_sr = [
    'skoci lijevo dva puta i hodaj desno',
    'trci nasuprotno desno tri puta',
    'gledaj naokolo lijevo i skoci',
]
examples_en = [
    'jump left twice and walk right',
    'run opposite right thrice',
    'look around left and jump',
]
if language == 'sr':
    examples = examples_sr 
else:
    examples = examples_en

for cmd in examples:
    pred = predict(
        cmd, model, tokenizer,
        prefix         = cfg['model']['prefix'],
        max_input_len  = cfg['model']['max_input_len'],
        max_target_len = cfg['model']['max_target_len'],
        device         = DEVICE,
        num_beams      = cfg['model']['num_beams'],
        bad_word_ids   = bad_word_ids,
    )
    print(f'Command : {cmd}')
    print(f'Actions : {pred}\n')


## 7. Evaluation

This section evaluates the model using both the Hugging Face `Trainer` API and a custom evaluation pipeline.

Evaluation metrics include:
- **Exact Match (EM)** – percentage of completely correct predictions  
- **Token Accuracy** – correctness at the token level  
- **Loss** – model error on the test set (computed during Trainer evaluation)

In [ ]:
from src.training.trainer import build_trainer

trainer = build_trainer(
    model             = model,
    tokenizer         = tokenizer,
    tokenized_dataset = tokenized_dataset,
    training_cfg      = cfg['training'],
    device_fp16       = (DEVICE == 'cuda'),
)

# Trainer metrics (exact match + token accuracy on full test set)
trainer_results = trainer.evaluate()
for k, v in trainer_results.items():
    print(f'  {k}: {v}')

In [ ]:
from src.evaluation.evaluation import evaluate_model, print_comparison
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive

results = evaluate_model(
    test_data, model, tokenizer, cfg, DEVICE,
    n=cfg['data']['n_eval']
)
print_comparison(results)

save_evaluation_results(results, split_name=split, output_dir='results')
copy_results_to_drive(output_dir="results")

### 7.1. Evaluation across SCAN splits

The model is evaluated on multiple SCAN splits to assess generalization across different task distributions, including:
- simple
- length
- addprim_jump
- template_around_right

For each split, the **exact match (constrained)** metric is reported on a subset of the test data.

In [ ]:
from src.data.load_data import load_scan
from src.evaluation.evaluation import evaluate_model
from src.evaluation.save_results import save_evaluation_results, copy_results_to_drive

splits_to_eval = ['simple', 'length', 'addprim_jump', 'template_around_right']

print('Multi-split evaluation (model trained on:', split, ')')
print('=' * 55)
for sp in splits_to_eval:
    try:
        _, test_split = load_scan(
            split     = sp,
            base_path = cfg['data']['scan_base_path'],
            lang      = language,
        )
        results = evaluate_model(test_split, model, tokenizer, cfg, DEVICE, n=200)
        print(f"  {sp:<30} exact match: {results['constrained_exact_match']:.2%}")
        save_evaluation_results(results, split_name=sp, output_dir='results')
        copy_results_to_drive(output_dir="results")
    except FileNotFoundError:
        print(f"  {sp:<30} file not found – skipping")